# 03 — Feature Engineering
Runs the rolling-window feature pipeline from `feature_engineering.py` and inspects the result: does each feature actually trend with RUL, or is it noise that a model will just overfit to?

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import config
import preprocessing
import feature_engineering as fe

plt.rcParams['figure.figsize'] = (10, 4)

## Run cleaning + labeling + feature engineering
Uses the exact same functions `train.py` and `compare_models.py` call — this notebook is for inspection, not a parallel implementation. If you change `feature_engineering.py`, re-run this cell to see the effect.

In [ ]:
raw_df = preprocessing.load_data()
clean_df = preprocessing.clean_data(raw_df)
labeled_df = preprocessing.label_rul(clean_df)
feat_df = fe.create_features(labeled_df)

feature_cols = fe.get_feature_columns(feat_df)
print(f'{len(feature_cols)} feature columns, {len(feat_df)} rows after dropping incomplete windows')
feat_df.head()

## Feature summary statistics

In [ ]:
feat_df[feature_cols].describe().T

## Correlation of each feature with RUL
Sorted — features near zero correlation are candidates to drop or investigate; they're adding dimensionality without signal. This is a linear correlation only, so a feature with real but nonlinear signal can still show low correlation here — treat this as a first pass, not a final verdict.

In [ ]:
target_corr = feat_df[feature_cols + [config.COL_RUL]].corr()[config.COL_RUL].drop(config.COL_RUL)
target_corr = target_corr.sort_values(key=abs, ascending=False)
target_corr

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
target_corr.plot(kind='barh', ax=ax)
ax.set_title('Feature correlation with RUL')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## Visualize top features vs. RUL
Scatter of the most-correlated features against RUL — check these actually look like usable degradation trends and not an artifact of how `RUL_CAP` / piecewise labeling was set.

In [ ]:
top_features = target_corr.abs().sort_values(ascending=False).head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, feat in zip(axes.ravel(), top_features):
    ax.scatter(feat_df[config.COL_RUL], feat_df[feat], s=4, alpha=0.3)
    ax.set_xlabel('RUL')
    ax.set_ylabel(feat)
plt.tight_layout()
plt.show()

## Save features for downstream notebooks/scripts
Writes to the same path `feature_engineering.run_feature_engineering()` uses, so `train.py` / `compare_models.py` pick up whatever was inspected here.

In [ ]:
feat_df.to_csv(config.FEATURES_DATA_PATH, index=False)
print(f'Saved to {config.FEATURES_DATA_PATH}')

## Next step
If features show plausible RUL trends, proceed to `05_model_baselines.ipynb`.